# Exercise 2, Part A: RAG over the Wflow paper

Exercise 1 extracted structured fields from a hydrology PDF. This part stays in **text**: **Retrieval-Augmented Generation (RAG)** answers questions from a document you index yourself — here the Wflow_sbm GMD paper.

You will:

1. Load the paper and split it into overlapping chunks.
2. Build two retrievers on the **same** chunks: keyword (token overlap) and semantic (embeddings + cosine).
3. Compare what each method returns for hydrology queries.
4. Generate answers with Claude or Mistral: no context vs keyword vs semantic.

We do **not** compute RAGAS or other automatic RAG scores. Judge the chunks and answers yourself.

Prerequisites: course `uv sync` / hydrology-env verify is green, and Foundry (or Anthropic) variables are in the repo-root `.env`. Never paste API keys into this notebook.

## Step 1: Imports and paths


In [12]:
import html
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from dllmforge.rag_preprocess_documents import PDFLoader, TextChunker
from IPython.display import Markdown, display

EXERCISE_DIR = Path.cwd()
if not (EXERCISE_DIR / "data" / "pdfs").exists():
    EXERCISE_DIR = Path.cwd() / "exercises" / "02_mcp_rag"
PDF_PATH = EXERCISE_DIR / "data" / "pdfs" / "gmd-17-3199-2024.pdf"
if not PDF_PATH.is_file():
    raise FileNotFoundError(f"Missing paper: {PDF_PATH}")

print(f"Exercise dir: {EXERCISE_DIR}")
print(f"PDF: {PDF_PATH.name}")


Exercise dir: c:\Users\deng_jg\work\16centralized_agents\hydrology_user_day_training\LLMs-Agentic-Workflows-in-Hydrology-Training\exercises\02_mcp_rag
PDF: gmd-17-3199-2024.pdf


## Step 2: LLM and embeddings

`PROVIDER` picks the generator: `anthropic` (Claude) or `mistral`. Embeddings always use the Foundry OpenAI-compatible endpoint (`DEPLOYMENT_TEXT_EMBEDDINGS_3_LARGE`).

In [13]:
PROVIDER = "anthropic"  # or "mistral"


def _load_env() -> None:
    env = find_dotenv(usecwd=True)
    if not env:
        for parent in [Path.cwd(), *Path.cwd().parents]:
            cand = parent / ".env"
            if cand.is_file():
                env = str(cand)
                break
    load_dotenv(env)


def foundry_anthropic_url() -> str | None:
    ep = (os.getenv("FOUNDRY_ENDPOINT") or "").strip().rstrip("/")
    if not ep or not os.getenv("FOUNDRY_KEY"):
        return None
    for suffix in ("/v1/messages", "/v1"):
        if ep.endswith(suffix):
            ep = ep[: -len(suffix)]
    return ep


def foundry_openai_url() -> str | None:
    if os.getenv("MISTRAL_ENDPOINT"):
        return os.environ["MISTRAL_ENDPOINT"].strip().rstrip("/")
    base = foundry_anthropic_url()
    if not base:
        return None
    return f"{base.split('/anthropic')[0]}/openai/v1"


def text_of(msg_or_content) -> str:
    # Claude returns a list of content blocks; keep only text.
    c = getattr(msg_or_content, "content", msg_or_content)
    if isinstance(c, str):
        return c
    parts = [
        b if isinstance(b, str) else b.get("text", "")
        for b in c or []
        if isinstance(b, str) or (isinstance(b, dict) and b.get("type") == "text")
    ]
    return "\n".join(p for p in parts if p)


def get_llm(provider: str):
    if provider == "anthropic":
        from langchain_anthropic import ChatAnthropic

        model = (
            os.getenv("AGENT_ANTHROPIC_MODEL")
            or os.getenv("DEPLOYMENT_OPUS_55")
            or os.getenv("ANTHROPIC_DEPLOYMENT_SONNET55")
            or "claude-sonnet-5-5"
        )
        foundry_url = foundry_anthropic_url()
        if foundry_url:
            return ChatAnthropic(
                model=model, max_tokens=4096, max_retries=3,
                base_url=foundry_url, api_key=os.getenv("FOUNDRY_KEY"),
            )
        if os.getenv("ANTHROPIC_API_KEY"):
            return ChatAnthropic(model=model, max_tokens=4096, max_retries=3)
        raise RuntimeError("Set FOUNDRY_ENDPOINT + FOUNDRY_KEY, or ANTHROPIC_API_KEY.")
    if provider == "mistral":
        from langchain_openai import ChatOpenAI

        url = foundry_openai_url()
        key = os.getenv("MISTRAL_KEY") or os.getenv("FOUNDRY_KEY")
        model = (
            os.getenv("DEPLOYMENT_MISTRAL_LARGE_4")
            or os.getenv("DEPLOYMENT_MISTRAL_MEDIUM_35")
            or "mistral-large-4"
        )
        if not url or not key:
            raise RuntimeError(
                "Mistral needs FOUNDRY_ENDPOINT + FOUNDRY_KEY "
                "(or MISTRAL_ENDPOINT + MISTRAL_KEY)."
            )
        return ChatOpenAI(
            model=model, base_url=url, api_key=key, temperature=0.0,
            top_p=1, max_tokens=4096, max_retries=3, use_responses_api=False,
        )
    raise ValueError(f"Unknown PROVIDER {provider!r}; use 'anthropic' or 'mistral'.")


_load_env()
llm = get_llm(PROVIDER)

from langchain_openai import OpenAIEmbeddings

embed_url = foundry_openai_url()
if not embed_url or not os.getenv("FOUNDRY_KEY"):
    raise RuntimeError("Embeddings need FOUNDRY_ENDPOINT + FOUNDRY_KEY.")
embeddings = OpenAIEmbeddings(
    model=os.getenv("DEPLOYMENT_TEXT_EMBEDDINGS_3_LARGE", "text-embedding-3-large"),
    base_url=embed_url,
    api_key=os.getenv("FOUNDRY_KEY"),
    check_embedding_ctx_length=False,
)
print(f"LLM: {PROVIDER}")
print(f"Embeddings: {embeddings.model}")


LLM: anthropic
Embeddings: text-embedding-3-large


## Step 3: Load the paper

`PDFLoader` (DLLMForge) extracts text per page with pypdf. Empty pages are skipped.

In [14]:
loader = PDFLoader()
pages, file_name, _metadata = loader.load(PDF_PATH)
print(f"{file_name}: {len(pages)} pages with text")
print("First page preview:")
print(pages[0][1][:400])


gmd-17-3199-2024.pdf: 36 pages with text
First page preview:
Geosci. Model Dev., 17, 3199–3234, 2024
https://doi.org/10.5194/gmd-17-3199-2024
© Author(s) 2024. This work is distributed under
the Creative Commons Attribution 4.0 License.
Model description paper
Wﬂow_sbm v0.7.3, a spatially distributed hydrological model:
from global data to local applications
Willem J. van Verseveld1, Albrecht H. Weerts1,2, Martijn Visser1, Joost Buitink1, Ruben O. Imhoff1,



## Step 4: Chunk the text

Two parameters matter:

- **`chunk_size`**: max characters per chunk. Smaller chunks retrieve more precisely but can split a sentence of context.
- **`overlap_size`**: characters copied from the end of one chunk into the next, so a split sentence is not lost.

We use the DLLMForge defaults (`1000` / `200`).

In [15]:
chunker = TextChunker(chunk_size=1000, overlap_size=200)
chunks = chunker.chunk_text(pages, file_name)
print(f"{len(chunks)} chunks")
print(f"Example chunk (page {chunks[0]['page_number']}):")
print(chunks[0]["text"][:300])


223 chunks
Example chunk (page 1):
Geosci. Model Dev., 17, 3199–3234, 2024
https://doi.org/10.5194/gmd-17-3199-2024
© Author(s) 2024. This work is distributed under
the Creative Commons Attribution 4.0 License.
Model description paper
Wﬂow_sbm v0.7.3, a spatially distributed hydrological model:
from global data to local applications



## Step 5: Two indexes on the same chunks

**Keyword search** scores a chunk by how often query terms appear in it (after lowercasing, dropping stopwords, and treating `kv₀` as `kv0`). It is strong on symbols and exact phrases, weak on paraphrases. The score is a **raw hit count** (0, 1, 2, …) with **no upper bound**: it grows with how often the terms appear and with chunk length.

**Semantic search** embeds each chunk once, then ranks by cosine similarity to the query embedding. It is strong when the question uses different words than the paper ("soil permeability" vs saturated hydraulic conductivity). After L2-normalizing both vectors, cosine is theoretically **−1 to 1** (1 = same direction, 0 = orthogonal). Top hydrology hits usually sit in a **narrow positive band** (often roughly 0.2–0.6 with `text-embedding-3-large`), not near 1.

The two scores are **not on the same scale**. Rank within each list; do not compare a keyword `11` to a semantic `0.45`.

In [16]:
_TOKEN = re.compile(r"[a-z0-9_]+", re.I)
_SUB = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
STOP = {"the", "a", "an", "is", "are", "was", "what", "how", "does", "do", "of", "in", "and", "or", "to", "for", "that"}


def tokenize(text: str) -> list[str]:
    return [t.lower() for t in _TOKEN.findall(text.translate(_SUB)) if len(t) > 1]


def keyword_search(query: str, docs: list[dict], k: int = 5) -> list[tuple[float, dict]]:
    q_terms = [t for t in tokenize(query) if t not in STOP]
    scored = []
    for doc in docs:
        hay = doc["text"].translate(_SUB).lower()
        score = float(sum(hay.count(t) for t in q_terms))
        scored.append((score, doc))
    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[:k]


def embed_texts(texts: list[str], batch_size: int = 32) -> np.ndarray:
    vecs: list[list[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        vecs.extend(embeddings.embed_documents(batch))
        print(f"embedded {min(i + batch_size, len(texts))}/{len(texts)}")
    return np.asarray(vecs, dtype=float)


def semantic_search(
    query: str, matrix: np.ndarray, docs: list[dict], k: int = 5
) -> list[tuple[float, dict]]:
    q = np.asarray(embeddings.embed_query(query), dtype=float)
    q = q / (np.linalg.norm(q) + 1e-12)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-12
    scores = (matrix / norms) @ q
    order = np.argsort(-scores)[:k]
    return [(float(scores[i]), docs[i]) for i in order]


vectors = embed_texts([c["text"] for c in chunks])
print(f"vector matrix: {vectors.shape}")


INFO:httpx:HTTP Request: POST https://dsd-llm-sweden.services.ai.azure.com/openai/v1/embeddings "HTTP/1.1 200 OK"


embedded 32/223


INFO:httpx:HTTP Request: POST https://dsd-llm-sweden.services.ai.azure.com/openai/v1/embeddings "HTTP/1.1 200 OK"


embedded 64/223


INFO:httpx:HTTP Request: POST https://dsd-llm-sweden.services.ai.azure.com/openai/v1/embeddings "HTTP/1.1 200 OK"


embedded 96/223


... (4 outputs left out)

INFO:httpx:HTTP Request: POST https://dsd-llm-sweden.services.ai.azure.com/openai/v1/embeddings "HTTP/1.1 200 OK"


embedded 192/223


INFO:httpx:HTTP Request: POST https://dsd-llm-sweden.services.ai.azure.com/openai/v1/embeddings "HTTP/1.1 200 OK"


embedded 223/223
vector matrix: (223, 3072)


## Step 6: Compare keyword vs semantic retrieval

Same `k=3`. Tables below: rank, score, page, snippet. Keyword scores are hit counts; semantic scores are cosine — do not compare the numbers across tables.

Typical split:

- `What is kv0?` — keyword often wins (exact symbol).
- `How permeable is the soil, and how does that control runoff?` — semantic often wins (paraphrase of Ksat / infiltration).
- `How is water routed down the hillslope and in the river?` — semantic often wins (same idea without the jargon).

In [17]:
def hits_table(hits: list[tuple[float, dict]]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "rank": i,
                "score": round(score, 3),
                "page": doc["page_number"],
                "snippet": " ".join(doc["text"].split())[:180],
            }
            for i, (score, doc) in enumerate(hits, 1)
        ]
    )


def show_rag(title: str, hits: list[tuple[float, dict]], answer: str | None = None) -> None:
    display(Markdown(f"## {title}"))
    display(hits_table(hits))
    for i, (score, doc) in enumerate(hits, 1):
        body = html.escape(doc["text"])
        display(
            Markdown(
                f"<details><summary><b>[{i}]</b> p.{doc['page_number']} · score {score:.3f}</summary>\n\n"
                f"<pre style='white-space:pre-wrap'>{body}</pre>\n\n</details>"
            )
        )
    if answer is not None:
        display(Markdown(f"### Answer\n\n{answer}"))


QUERIES = [
    "What is kv0?",
    "How permeable is the soil, and how does that control runoff?",
    "How is water routed down the hillslope and in the river?",
]
K = 3

retrieved = {}
for q in QUERIES:
    kw = keyword_search(q, chunks, k=K)
    sem = semantic_search(q, vectors, chunks, k=K)
    retrieved[q] = {"keyword": kw, "semantic": sem}
    display(Markdown(f"### Query: {q}"))
    display(Markdown("**Keyword** (hit count)"))
    display(hits_table(kw))
    display(Markdown("**Semantic** (cosine)"))
    display(hits_table(sem))


INFO:httpx:HTTP Request: POST https://dsd-llm-sweden.services.ai.azure.com/openai/v1/embeddings "HTTP/1.1 200 OK"


### Query: What is kv0?

**Keyword** (hit count)

,rank,score,page,snippet
0,1,4.0,12,c conductivity) for the bottom soil layernb an...
1,2,3.0,9,"ttransfer, pot,n=Ktvz,n (θtn−θr θs−θr )cn , (4..."
2,3,3.0,31,W. J. van Verseveld et al.: Wﬂow_sbm v0.7.3 32...


**Semantic** (cosine)

,rank,score,page,snippet
0,1,0.426,20,Kv0 The wﬂow_sbm parameter fKv controls the ex...
1,2,0.369,20,peaks and higher under baseﬂow conditions comp...
2,3,0.349,20,w avalanche routine) model is enabled. For bot...


... (8 outputs left out)

**Keyword** (hit count)

,rank,score,page,snippet
0,1,27.0,30,3228 W. J. van Verseveld et al.: Wﬂow_sbm v0.7...
1,2,21.0,8,le [mmt−1] at time step t into the soil (throu...
2,3,21.0,10,"torageStunsat, 1 at time stept: Stdeﬁcit=(θs−θ..."


**Semantic** (cosine)

,rank,score,page,snippet
0,1,0.544,3,ine are transferred to the optional snow (base...
1,2,0.533,4,and leakage result in a ﬂux from the saturated...
2,3,0.509,13,2 times half of the river bankfull depth (hban...


## Step 7: Generate answers

One paraphrase question, three generations:

1. **No context** — parametric memory only (may be fluent and uncited).
2. **Keyword chunks**.
3. **Semantic chunks**.

The system prompt forbids answering outside the retrieved context. If the chunks are wrong, a good model should say it does not know rather than invent Ksat values.

Expand a chunk (`details`) to read the text that actually went into the prompt.

In [18]:
QUESTION = "How permeable is the soil, and how does that control runoff?"


def answer_with_chunks(question: str, hits: list[tuple[float, dict]]) -> str:
    context = "\n\n".join(
        f"[{doc['file_name']} p.{doc['page_number']}]\n{doc['text']}" for _, doc in hits
    )
    msg = llm.invoke(
        [
            (
                "system",
                "You are a hydrologist. Answer only from the provided context. "
                "If the context does not contain the answer, say you do not know. "
                "Cite file name and page.",
            ),
            ("human", f"Question: {question}\n\nContext:\n{context}"),
        ]
    )
    return text_of(msg)


def answer_no_context(question: str) -> str:
    msg = llm.invoke(
        [
            ("system", "You are a hydrologist."),
            ("human", question),
        ]
    )
    return text_of(msg)


kw_hits = retrieved[QUESTION]["keyword"]
sem_hits = retrieved[QUESTION]["semantic"]

display(Markdown(f"## No context\n\n{answer_no_context(QUESTION)}"))
show_rag("Keyword", kw_hits, answer_with_chunks(QUESTION, kw_hits))
show_rag("Semantic", sem_hits, answer_with_chunks(QUESTION, sem_hits))


INFO:httpx2:HTTP Request: POST https://dsd-llm-sweden.services.ai.azure.com/anthropic/v1/messages "HTTP/1.1 200 OK"


## No context

# Soil Permeability and Runoff

I don't know which site you're asking about, so I can't give you a single number. Permeability can vary by more than 1,000× between soil types and even across one field. Below is how to estimate it and how it controls runoff. If you share details about your site, I can narrow it down.

## 1. Typical permeability by soil texture

Permeability is usually expressed as **saturated hydraulic conductivity (Ksat)**: the rate water moves through soil once it is fully wetted.

| Texture | Typical Ksat (mm/hr) | Relative behavior |
|---|---|---|
| Sand | ~100–200+ | Absorbs almost any rainfall |
| Loamy sand | ~30 | Very high infiltration |
| Sandy loam | ~10 | High |
| Loam / silt loam | ~3–7 | Moderate |
| Clay loam / silty clay loam | ~1 | Low |
| Clay | ~0.3 or less | Very low, runoff-prone |

*(Values approximate, after Rawls et al., 1982. Field values often differ by an order of magnitude.)*

**USDA-NRCS Hydrologic Soil Groups** classify soils by the least permeable layer near the surface:
- **A**: >40 µm/s (~>145 mm/hr). Low runoff potential.
- **B**: 10–40 µm/s (~36–145 mm/hr).
- **C**: 1–10 µm/s (~3.6–36 mm/hr).
- **D**: <1 µm/s (~<3.6 mm/hr), or a shallow water table or shallow bedrock. High runoff potential.

You can look up your site's group in the **NRCS Web Soil Survey** (US) or an equivalent national soil database.

## 2. How permeability controls runoff

The key comparison is **rainfall intensity versus the soil's infiltration capacity.**

### Infiltration-excess (Hortonian) overland flow
When rain falls faster than the soil can absorb it, the excess runs off the surface.
- Example: a 25 mm/hr storm on clay loam (~1 mm/hr) produces runoff within minutes. The same storm on sandy loam may produce almost none.
- This mechanism dominates in arid lands, on crusted or compacted soils, in urban areas, and during intense convective storms.

### Saturation-excess overland flow
Even permeable soil generates runoff once the profile fills up, typically where a shallow water table or a restrictive layer (clay pan, bedrock) lies beneath. Any further rain, regardless of intensity, runs off.
- This mechanism dominates in humid, vegetated catchments, valley bottoms, and near streams. The saturated area expands during the storm (the "variable source area" concept).

### Subsurface stormflow
In permeable soils over a less permeable layer, water moves laterally downslope through the soil and macropores. It reaches streams more slowly than overland flow. This gives a delayed, attenuated hydrograph and a higher share of baseflow.

### Net effect on the hydrograph
- **Low permeability:** fast response, high peak flow, large runoff volume, more erosion, less groundwater recharge.
- **High permeability:** slow, damped response, lower peaks, more recharge and baseflow.

## 3. Factors that modify "textbook" permeability

- **Antecedent moisture:** wet soils have less storage and lower infiltration rates. This is the largest storm-to-storm control.
- **Surface sealing and crusting:** raindrop impact on bare silty soils can drop surface infiltration to a fraction of the subsoil's rate.
- **Compaction:** from machinery, livestock, or construction.
- **Macropores:** root channels, worm burrows, and cracks can raise effective infiltration far above matrix Ksat. Shrink-swell clays crack when dry and seal when wet.
- **Vegetation and organic matter:** improve structure, slow flow, and add interception.
- **Frozen ground:** can make even sandy soil nearly impermeable.
- **Water repellency:** after fire or in some dry organic soils.
- **Layering:** the least permeable layer often governs, not the surface texture.
- **Slope and soil depth:** thin soils on steep slopes saturate quickly.

## 4. Measuring it at your site

- **Field:** double-ring or single-ring infiltrometer, tension (disc) infiltrometer, Guelph permeameter, or a rainfall simulator.
- **Lab:** constant- or falling-head permeameter on undisturbed cores.
- **Estimation:** pedotransfer functions from texture, bulk density, and organic matter (e.g., Rosetta).
- **For runoff modeling:** convert soil group, land use, and condition to an **SCS Curve Number**, or use Green-Ampt or Horton infiltration parameters.

---

**To give you a site-specific answer, tell me:**
1. Location, or soil series/texture if known
2. Land use and cover (forest, crop, pasture, urban, bare)
3. Slope, and depth to bedrock or water table if known
4. Your purpose (stormwater design, flood estimate, erosion, recharge, septic siting, etc.)
5. Any infiltration test data you already have

With that, I can estimate infiltration rates, a curve number, and likely runoff for your design storm.

INFO:httpx2:HTTP Request: POST https://dsd-llm-sweden.services.ai.azure.com/anthropic/v1/messages "HTTP/1.1 200 OK"


## Keyword

,rank,score,page,snippet
0,1,17.0,10,"layer Stunsat, 1 (Eq. 55), and the remain- ing..."
1,2,17.0,31,W. J. van Verseveld et al.: Wﬂow_sbm v0.7.3 32...
2,3,15.0,10,estricted to the upper layer. As with the case...


<details><summary><b>[1]</b> p.10 · score 17.000</summary>

<pre style='white-space:pre-wrap'>layer Stunsat, 1 (Eq. 55), and the remain-
ing potential soil evaporationEtremainder, soil is determined as
follows:
Stunsat, 1=Stunsat, 1−Etact, soil, (65)
Etremainder, soil=Etpot, soil−Etact, soil. (66)
When the soil bucket in wﬂow_sbm is split up into differ-
ent layers, soil evaporation Etact, soil, sat [mmt−1] from the
saturated store is possible when the water table zt−1watertable
is present in the upper soil layer with actual thickness
z1, thickness [mm], and it is calculated as follows:
Etact, soil, sat= min(Etremainder, soilz1, thickness−zt−1watertablez1, thickness ,
(z1, thickness−zt−1watertable)(θs−θr)), (67)
withθs andθr as previously deﬁned. In the case of a single
soil layer or when zt−1watertable is not present in the upper soil
layer,Etact, soil, sat is set at zero.Etact, soil, sat is subtracted from
the saturated store (at the previous time step):
Stsat=St−1sat −Etact, soil, sat.</pre>

</details>

... (6 outputs left out)

<details><summary><b>[1]</b> p.3 · score 0.452</summary>

<pre style='white-space:pre-wrap'>ine are transferred to the optional
snow (based on the HBV-96 hydrological model concept;
Bergström, 1992) and glacier routines (based on the HBV-
light degree-day-based model; Seibert et al., 2018). The soil
in every grid cell is considered a single bucket, divided into a
saturated and unsaturated store, with the option to divide the
soil column into different layers. Available inﬁltration (stem-
ﬂow and throughfall not converted into snow, including melt-
water) inﬁltrates into the soil or becomes direct runoff based
on the river fraction or open-water (excluding rivers) frac-
tion. Soil inﬁltration is determined separately for the paved
and nonpaved areas, as these have different inﬁltration ca-
pacities. Naturally, only the water that can be stored in the
soil can inﬁltrate. If not all water can inﬁltrate, this is added
as saturation excess water to the runoff routing scheme for
overland ﬂow.</pre>

</details>

<details><summary><b>[2]</b> p.3 · score 0.442</summary>

<pre style='white-space:pre-wrap'>cities. Naturally, only the water that can be stored in the
soil can inﬁltrate. If not all water can inﬁltrate, this is added
as saturation excess water to the runoff routing scheme for
overland ﬂow. Inﬁltration excess occurs when the inﬁltra-
tion capacity is smaller than the available inﬁltration rate,
and this amount of water is also added to the runoff rout-
ing scheme for overland ﬂow. An exponential decay of the
https://doi.org/10.5194/gmd-17-3199-2024 Geosci. Model Dev., 17, 3199–3234, 2024</pre>

</details>

<details><summary><b>[3]</b> p.8 · score 0.434</summary>

<pre style='white-space:pre-wrap'>ertable [mm]
such that the value ofSsat is given by
Ssat=(zsoil−zwatertable)(θs−θr), (40)
where θs and θr are the saturated and residual soil water
contents, respectively, both expressed in mm mm −1. The
amount of water that can inﬁltrate is a function of the in-
ﬁltration capacitycinﬁltration, paved [mmt−1] of the compacted
soil (or paved area) fraction ( fpaved [–]) of each grid cell,
the inﬁltration capacity cinﬁltration, unpaved [mmt−1] of the
non-compacted soil fraction (or unpaved area) ((1−fpaved))
of each grid cell, the initial storage capacity of the unsat-
urated zone Stunsat,max [mm] (Eq. 43) at time step t and
an optional reduction factor ffrozen applied to the inﬁltra-
tion capacity when snow is modelled, and the model set-
ting soilinfreduction is set to true. The parameter ffrozen de-
pends on the near-surface soil temperature, which is mod-
elled based on the approach of Wigmosta et al.</pre>

</details>

### Answer

**Permeability:** The excerpts don't give a value for how permeable the soil is. They describe how a model represents infiltration, but they don't report hydraulic conductivity or infiltration-capacity values for any site. I can't answer that part from this context.

**How infiltration controls runoff in the model:**

- **Soil structure:** Each grid cell's soil is a single bucket with a saturated and an unsaturated store. It can optionally be split into layers. (gmd-17-3199-2024.pdf, p.3)
- **Direct runoff:** Water that is available for infiltration either enters the soil or becomes direct runoff, depending on the river fraction or open-water fraction of the cell. (p.3)
- **Paved vs. unpaved areas:** Infiltration is calculated separately for the two, because they have different infiltration capacities. (p.3) The amount that can infiltrate depends on:
  - the infiltration capacity of the compacted (paved) fraction, *c_infiltration,paved*, weighted by *f_paved*;
  - the infiltration capacity of the non-compacted (unpaved) fraction, *c_infiltration,unpaved*, weighted by *(1 − f_paved)*;
  - the available storage capacity of the unsaturated zone at that time step; and
  - an optional frozen-soil reduction factor, *f_frozen*. This applies when snow is modelled and the *soilinfreduction* setting is true, and it depends on near-surface soil temperature. (p.8)
- **Two runoff-generating mechanisms**, both sent to the overland flow routing scheme:
  - **Saturation excess:** Only water the soil can store can infiltrate. Any surplus becomes overland flow. (p.3)
  - **Infiltration excess:** When the infiltration capacity is smaller than the available infiltration rate, the excess also becomes overland flow. (p.3)
- **Storage depends on soil properties:** The saturated store is calculated as S_sat = (z_soil − z_watertable)(θs − θr), using the saturated and residual water contents. These partly determine how much water the soil can hold before saturation excess occurs. (p.8)

In short, lower infiltration capacity, more paved area, frozen soil, or limited unsaturated storage all increase overland flow in the model. Actual permeability values would have to come from the model's parameter maps or another source.

## Step 8: Your turn

Change `MY_QUESTION` and re-run. Try a symbol (`kv0`, `Ksat`) and a paraphrase of the same idea. 

In [19]:
MY_QUESTION = "What is the role of capillary rise in wflow_sbm?"

kw = keyword_search(MY_QUESTION, chunks, k=K)
sem = semantic_search(MY_QUESTION, vectors, chunks, k=K)
show_rag("Keyword", kw)
show_rag("Semantic", sem, answer_with_chunks(MY_QUESTION, sem))


INFO:httpx:HTTP Request: POST https://dsd-llm-sweden.services.ai.azure.com/openai/v1/embeddings "HTTP/1.1 200 OK"


## Keyword

,rank,score,page,snippet
0,1,4.0,11,"xcess water rate at the surfaceRtexcess, unsat..."
1,2,4.0,31,W. J. van Verseveld et al.: Wﬂow_sbm v0.7.3 32...
2,3,3.0,12,3210 W. J. van Verseveld et al.: Wﬂow_sbm v0.7...


<details><summary><b>[1]</b> p.11 · score 4.000</summary>

<pre style='white-space:pre-wrap'>xcess water
rate at the surfaceRtexcess, unsat [mmt−1]. The actual inﬁltra-
tion rateFtact [mmt−1] is then calculated as follows:
Ftact=Fttotal−Rtexcess,unsat, (80)
with Fttotal as previously deﬁned. The rate of water that
cannot inﬁltrate due to saturated soil conditions Ftexcess, sat
[mmt−1] is determined as
Ftexcess,sat=Ftavailable−Fttotal−Ftexcess+Rtexcess,unsat, (81)
withFtavailable,Fttotal andFtexcess as previously deﬁned.
Capillary rise in wﬂow_sbm is determined when an unsat-
urated zone occurs in the soil column at time stept based on
the water table depth at the previous time step ( zt−1watertable&gt;
0), using the following approach. First a maximum capillary
riseCtmax [mmt−1] at time stept is determined from the min-
imum of Ktvz,m (Eq. 50); the actual transpiration rate from
the unsaturated store ∑mn=1Ettrans, unsat,n for m unsaturated
soil layers; Stsat (Eq. 72) and the unsaturated store capacity
Stunsat, max, which is based on Stsat (Eq.</pre>

</details>

<details><summary><b>[2]</b> p.31 · score 4.000</summary>

<pre style='white-space:pre-wrap'>W. J. van Verseveld et al.: Wﬂow_sbm v0.7.3 3229
Table A2. Wﬂow_sbm model parameters and forcing.
Symbol Description Unit Wﬂow.jl name Default value
P Precipitation mm t−1 precipitation –
Epot, total Potential evapotranspiration mm t−1 potential_evaporation –
Tair Mean air temperature °C temperature –
zsoil Soil depth mm soilthickness 2000.0
θs Saturated soil water content mm mm −1 θs 0.6
θr Residual soil water content mm mm −1 θr 0.01
fpaved Fraction of compacted soil (or paved) – pathfrac 0.01
fopen water Open-water fraction (excluding rivers) – waterfrac 0.0
friver River fraction – riverfrac –
cinﬁltration, unpaved Inﬁltration capacity of non-compacted soil mm t−1 infiltcapsoil 100.0 mm d−1
cinﬁltration, paved Inﬁltration capacity of compacted soil mm t−1 infiltcappath 10.0 mm d−1
zrooting Rooting depth mm rootingdepth 750.0
Esat
Psat Gash interception model parameter – e_r 0.1
LAI Leaf area index m 2 m−2 leaf_area_index –
Sleaf Speciﬁc leaf storage mm sl –
Scanopy, max Canopy storage capacity mm cmax 1.0
fcanopygap Canopy gap fraction – canopygapfraction 0.1
Swood, max Storage capacity, woody parts of vegetation mm swood –
k Extinction coefﬁcient – kext –
crd Model parameter controlling the sigmoid function, for the frac-
tion of wet roots
mm−1 rootdistpar −500.0
zcap, maxdepth Critical water depth beyond which capillary rise ceases mm cap_hmax 2000.0
ncap Empirical coefﬁcient controlling capillary rise – cap_n 2.0
Kv0 Vertical saturated hydraulic conductivity at the soil surface mm t−1 kv0 3000.0 mm d−1
fKv Scaling parameter for vertical saturated hydraulic conductivity mm −1 f 0.001
cn Brooks–Corey power coefﬁcient – c 10.0
hb Air entry value cm hb 10.0
Lmax Maximum allowed leakage mm t−1 maxleakage 0.0 mm d−1
fKh0 Multiplication factor applied to Kv0 (for lateral subsurface
ﬂow)
– – 1.0
fKv,n Multiplication factor (correcting vertical saturated hydraulic
conductivity)
– kvfrac 1.0
sddf Degree-day-melt factor snow mm t−1 °C−1 cfmax 3.75653 mm d−1 °C−1
sfall,T threshold Temperature threshold for snowfall °C tt 0.0
sfall,T interval Temperature threshold interval for snowfall °C tti 1.0
smelt,T threshold Temperature threshold for snowmelt °C ttm 0.0
swhc Water-holding capacity of snow – whc 0.1
gmelt,T threshold Temperature threshold for glacier melt °C g_tt 0.0
gddf Degree-day-melt factor glacier mm t−1 °C−1 g_cfmax 3.0 mm d−1 °C−1
fglacier Fraction covered by a glacier – glacierfrac 0.0
Sglacier Glacier storage mm glacierstore 5500.0
gsnow to ﬁrn Fraction of snow that is converted into ﬁrn t−1 g_sifrac 0.001 d−1
wsoil Weighting coefﬁcient for near-surface soil temperature t−1 w_soil 0.1125 d−1
fred, frozen Controlling inﬁltration reduction factor – cf_soil 0.038
cland slope Slope of the land surface m m −1 βl –
criver slope Slope of river m m −1 sl –
Ares Reservoir area m 2 area –
Qmin req.</pre>

</details>

<details><summary><b>[3]</b> p.12 · score 3.000</summary>

<pre style='white-space:pre-wrap'>3210 W. J. van Verseveld et al.: Wﬂow_sbm v0.7.3
equation (e.g. Zammouri, 2001; Yang et al., 2011; Wang et
al., 2016):
Ctact=



Ctmax
(
1− zt−1watertablezcap, maxdepth
)ncap
if(zt−1watertable&gt;z rooting) and
(zt−1watertable&lt;z cap, maxdepth)
0 otherwise ,
(84)
where Ctact [mmt−1] is the capillary rate at time step t;
zcap, maxdepth [mm] is the critical water depth beyond which
capillary rise ceases; ncap [–] is an empirical coefﬁcient re-
lated to soil properties and climate, generally set between 1–
3; andzt−1watertable andzrooting are as previously deﬁned. When
the soil bucket in wﬂow_sbm is split up into different layers,
Ctact is divided over the different unsaturated soil layers, from
the bottom to the top unsaturated soil layer, without exceed-
ing the saturated water contentθs.
2.4.4 Leakage
In wﬂow_sbm it is possible to have leakage Lt [mmt−1] at
time step t from the saturated store Stsat (Eq.</pre>

</details>

... (3 outputs left out)

<details><summary><b>[1]</b> p.11 · score 0.655</summary>

<pre style='white-space:pre-wrap'>xcess water
rate at the surfaceRtexcess, unsat [mmt−1]. The actual inﬁltra-
tion rateFtact [mmt−1] is then calculated as follows:
Ftact=Fttotal−Rtexcess,unsat, (80)
with Fttotal as previously deﬁned. The rate of water that
cannot inﬁltrate due to saturated soil conditions Ftexcess, sat
[mmt−1] is determined as
Ftexcess,sat=Ftavailable−Fttotal−Ftexcess+Rtexcess,unsat, (81)
withFtavailable,Fttotal andFtexcess as previously deﬁned.
Capillary rise in wﬂow_sbm is determined when an unsat-
urated zone occurs in the soil column at time stept based on
the water table depth at the previous time step ( zt−1watertable&gt;
0), using the following approach. First a maximum capillary
riseCtmax [mmt−1] at time stept is determined from the min-
imum of Ktvz,m (Eq. 50); the actual transpiration rate from
the unsaturated store ∑mn=1Ettrans, unsat,n for m unsaturated
soil layers; Stsat (Eq. 72) and the unsaturated store capacity
Stunsat, max, which is based on Stsat (Eq.</pre>

</details>

<details><summary><b>[2]</b> p.4 · score 0.631</summary>

<pre style='white-space:pre-wrap'>3202 W. J. van Verseveld et al.: Wﬂow_sbm v0.7.3
saturated hydraulic conductivity with soil depth is assumed.
Transfer of water to the unsaturated store and to the saturated
store is based on Brooks and Corey (1964) when the soil col-
umn is divided into different layers, and in the case of one
soil layer the original Topog_SBM vertical transfer formula-
tion can also be used. Part of the water evaporates through
soil evaporation, transpiration that is ﬁrst derived from the
saturated store if roots intersect with the saturated store and
then from the unsaturated store, and open-water (excluding
rivers) and river evaporation. Besides transpiration, capillary
rise and leakage result in a ﬂux from the saturated store to the
unsaturated store and outside of the model domain, respec-
tively. The kinematic-wave approach is used to route subsur-
face ﬂow laterally.</pre>

</details>

<details><summary><b>[3]</b> p.2 · score 0.625</summary>

<pre style='white-space:pre-wrap'>s routed for example over the land or
river domain. This paper presents the wﬂow_sbm model con-
ﬁguration that makes use of the kinematic-wave approach
for river, overland and lateral subsurface ﬂow. It is part
of the open-source distributed hydrological model platform
Wﬂow.jl (van Verseveld et al., 2024) developed at Deltares.
Wﬂow_sbm strikes a balance between low-resolution, low-
complexity and high-resolution, high-complexity hydrologi-
cal models, giving an answer to most of the aforementioned
challenges. In this model, the soil part is largely based on
Topog_SBM (Vertessy and Elsenbeer, 1999), with gravity-
based inﬁltration and vertical ﬂow through the soil column
as well as capillary rise representing a simpliﬁed version
of Richards’ equation.</pre>

</details>

### Answer

## Role of capillary rise in wflow_sbm

**1. It is part of the model's simplified Richards' equation.**
The wflow_sbm soil component is largely based on Topog_SBM. Gravity-based infiltration, vertical flow through the soil column and capillary rise together represent "a simplified version of Richards' equation" (gmd-17-3199-2024.pdf, p.2).

**2. It moves water upward from the saturated store to the unsaturated store.**
Transpiration, capillary rise and leakage all draw on the saturated store. Capillary rise is the flux that carries water from the saturated store into the unsaturated store. Leakage is different: it removes water from the saturated store to outside the model domain (gmd-17-3199-2024.pdf, p.4).

**3. It is only computed under certain conditions.**
Capillary rise is determined only when an unsaturated zone exists in the soil column at time step *t*. This is judged from the water table depth at the previous time step (zᵗ⁻¹_watertable > 0) (gmd-17-3199-2024.pdf, p.11).

**4. It is limited by a maximum rate.**
A maximum capillary rise Cᵗ_max [mm t⁻¹] is set as the minimum of four terms (gmd-17-3199-2024.pdf, p.11):
- the vertical hydraulic conductivity Kᵗ_vz,m (Eq. 50);
- the actual transpiration rate from the unsaturated store, summed over the *m* unsaturated soil layers (Σ Eᵗ_trans,unsat,n);
- the saturated store Sᵗ_sat (Eq. 72);
- the unsaturated store capacity Sᵗ_unsat,max, which is based on Sᵗ_sat.

Including transpiration from the unsaturated store in this limit links capillary rise to the water demand created by plant transpiration.

**What the context does not cover:** The excerpt is cut off after the definition of the maximum capillary rise. I cannot say how the actual capillary rise is calculated from Cᵗ_max, for example any depth-dependent reduction (gmd-17-3199-2024.pdf, p.11).

## Takeaways

- **Keyword search** is good for codes and symbols that actually appear (`kv0`, `kinematic wave`).
- **Semantic search** is good when you ask in different words than the paper ("soil permeability" vs saturated hydraulic conductivity).